# Tutorial 06d — Fused Attention **Backward**, for real: Triton on CUDA, Ampere/Ada only

> A companion to **`tutorials_jupyter/06-fused-attention.ipynb`** (OpenAI's Triton
> **FlashAttention-2**). **06a** unpacked the forward. **06b** unpacked the backward *on paper*.
> **06c** stripped the forward to CUDA + Ampere/Ada and ran it.
>
> ### This notebook is 06b's runnable twin.
>
> Same four kernels — `_attn_bwd_preprocess`, `_attn_bwd_dkdv`, `_attn_bwd_dq`, `_attn_bwd` — but
> promoted from prose to **real Triton on your GPU**: launched, checked against `torch.autograd`,
> deliberately broken to see which gradient moves, and benchmarked.
>
> **Persona:** entry-to-intermediate practitioner (you know PyTorch, the GPU memory hierarchy, and
> roughly what a Triton kernel is). Same voice as **06a**/**06b**/**06c**.

### Prerequisites

- **06b**, all of it. We are *not* re-deriving the five gradients, the `D = rowsum(O ∘ dO)` collapse,
  recompute-from-`M`, or the dK/dV-over-keys vs dQ-over-queries split. You should already know why
  `dS = P ∘ (dP − D)` and why `dQ` reduces along the *other* axis from `dK`/`dV`.
- **06c**, at least §7 (the stripped forward). We reuse its `_attn_fwd` verbatim to produce the `O`
  and `M` this backward consumes.


### A confession about the title

This notebook is called *minimal … Ampere/Ada only*, which promises a **strip** — the deletion of
architecture-specific branches, as 06c did for the forward.

**There is nothing to strip.** §1 measures it: the tutorial's backward contains **zero** architecture
gates. No `is_hip()`, no TMA descriptors, no `warp_specialize`, no FP8, no Blackwell branch. It has
always used plain pointer arithmetic. 06c's §3 predicted exactly this, and here is the receipt.

So this chapter answers the harder question that discovery leaves behind:

> **If the backward has no architecture branches, what *is* Ampere/Ada-specific about it?**

The answer is not a line of source. It is a **shared-memory budget** — and it is severe enough that
the tutorial's own test suite **cannot run its backward at `HEAD_DIM=128` on any consumer Ampere or
Ada GPU**. We hit that wall in §10, understand it in §11, and fix it.

### Learning objectives

By the end you will be able to:

- **Audit** a kernel for architecture gates mechanically, and trust the answer.
- Launch all four backward kernels and verify their gradients against `torch.autograd`.
- **Diagnose a wrong gradient from its ratio**: recognise `×1.4427` as a missing `dq *= LN2` and
  `×2.0` as a missing `dk *= sm_scale`.
- Compute a kernel's **shared-memory footprint**, and predict from it whether it will even *launch*.
- Explain why `num_stages=3` beats `num_stages=5` by **21%**, using **blocks resident per SM**.
- Say honestly where this backward stands against `torch`'s own — and why.


### How to use this notebook

Like **06c** and unlike 06a/06b, **the kernels here really run.** With a CUDA GPU and `triton` +
`torch` installed, every code cell executes: it compiles the backward, checks it against PyTorch
autograd, breaks it on purpose, measures its shared memory, and benchmarks it.

Without a GPU nothing breaks. Kernel cells still **define** (via a no-op shim in §2), all five
exercises still run — they are pure NumPy/Python — and every cell prints the numbers **recorded on
the reference machine**: an **RTX 4080 SUPER (sm_89, Ada), Triton 3.7.0, torch 2.12.0+cu130**.

### Which GPUs is this for?

The hardware fact that decides this chapter is the **maximum shared memory per thread block**. Keep
this table open — §10 is entirely about it.

| Architecture | cc | Max shared memory / thread block | Example cards | The backward at `HEAD_DIM=128` |
|---|---|---|---|---|
| Ampere (datacenter) | 8.0 | 163 KB | A100 | fits — big enough to hide the problem |
| Ampere (GA10x) | 8.6 | **99 KB** | RTX 30xx, A10 | **will not launch** |
| Ada Lovelace | 8.9 | **99 KB** | RTX 40xx, L40S | **will not launch** |
| Hopper | 9.0 | 227 KB | H100, H200 | fits — and is what upstream was tuned on |
| Blackwell (datacenter) | 10.0 | 227 KB | B200 | fits |
| Blackwell (GeForce) | 12.0 | **99 KB** | RTX 50xx | **inherits the problem** |

Sources: [CUDA Programming Guide — Technical Specifications per Compute Capability](https://docs.nvidia.com/cuda/cuda-programming-guide/05-appendices/compute-capabilities.html),
[Ampere Tuning Guide](https://docs.nvidia.com/cuda/ampere-tuning-guide/index.html),
[Ada Tuning Guide](https://docs.nvidia.com/cuda/ada-tuning-guide/index.html),
[Blackwell Tuning Guide](https://docs.nvidia.com/cuda/blackwell-tuning-guide/index.html).


## 0. A map of where the architecture hides

In the **forward**, hardware-specificity is *lexical*: you can see it, as `if` statements, and delete
it. That was 06c.

In the **backward**, hardware-specificity is *resource-shaped*: invisible in the source, decided by
the compiler's shared-memory allocation — and it does not degrade gracefully. It refuses to launch.

```mermaid
graph TD
  subgraph FWD["_attn_fwd — 273 lines, 33 gate references"]
    A["is_hip()?"] --> B["supports_host_descriptor()?"]
    B --> C["is_blackwell() and warp_specialize?"]
    C --> D["FP8_OUTPUT?"]
    D --> E["compiles to the code YOUR card runs"]
  end
  subgraph BWD["_attn_bwd — 297 lines, 0 gate references"]
    F["plain pointer arithmetic,<br/>identical source on every GPU"] --> G{"does its shared memory<br/>fit YOUR block limit?"}
    G -- "yes (HEAD_DIM=64: 55 KB)" --> H["runs"]
    G -- "no (HEAD_DIM=128: 107 KB > 99 KB)" --> I["OutOfResources<br/>refuses to launch"]
  end
  style E fill:#c6f6d5,stroke:#2f855a
  style H fill:#c6f6d5,stroke:#2f855a
  style I fill:#fed7d7,stroke:#c53030
```

The rest of the notebook walks the right-hand box: build it (§3–§7), trust it (§8–§9), then find its
ceiling (§10–§12).


## 1. Audit it yourself — don't take my word for it

A claim like *"the backward has no architecture gates"* is exactly the kind of thing that is easy to
assert and easy to get wrong. So let's **count**, mechanically, over the tutorial's real source.

We split `06-fused-attention.py` into four regions — the forward kernels, the forward launcher, the
backward kernels, the backward launcher — and grep each for every token that names a hardware
capability.


In [ ]:
import json
import re
from pathlib import Path

GATE_TOKENS = ["is_hip", "is_cuda", "supports_host_descriptor", "is_hopper", "is_blackwell",
               "warp_specialize", "make_tensor_descriptor", "TensorDescriptor", "float8e5", "maxnreg"]

REGIONS = [                                   # (label, start marker, end marker)
    ("_attn_fwd* kernels", "def _attn_fwd_inner", "def _attn_bwd_preprocess"),
    ("_attention.forward", "    def forward(ctx", "    def backward(ctx"),
    ("_attn_bwd* kernels", "def _attn_bwd_preprocess", "class _attention"),
    ("_attention.backward", "    def backward(ctx", "attention = _attention.apply"),
]

CANDIDATES = [Path("06-fused-attention.ipynb"),
              Path("tutorials_jupyter/06-fused-attention.ipynb"),
              Path("../tutorials_jupyter/06-fused-attention.ipynb")]


def upstream_source():
    """Concatenate the code cells of the upstream tutorial notebook, if we can find it."""
    for p in CANDIDATES:
        if p.exists():
            nb = json.loads(p.read_text())
            return "".join("".join(c["source"]) if isinstance(c["source"], list) else c["source"]
                           for c in nb["cells"] if c["cell_type"] == "code")
    return None


def audit(src):
    rows = []
    for label, start, end in REGIONS:
        region = src[src.index(start):src.index(end)]
        hits = {t: len(re.findall(re.escape(t), region)) for t in GATE_TOKENS}
        which = ", ".join(t for t, n in hits.items() if n) or "—"
        rows.append((label, region.count("\n"), sum(hits.values()), which))
    return rows


src = upstream_source()
if src is None:
    print("Could not find 06-fused-attention.ipynb; showing the recorded audit (Triton 3.7.0):\n")
    rows = [("_attn_fwd* kernels", 203, 16, "is_hip, is_cuda, supports_host_descriptor, warp_specialize, ..."),
            ("_attention.forward", 70, 17, "is_hip, supports_host_descriptor, is_hopper, is_blackwell, ..."),
            ("_attn_bwd* kernels", 254, 0, "—"),
            ("_attention.backward", 43, 0, "—")]
else:
    rows = audit(src)

print(f"{'region':<22}{'lines':>7}{'gate refs':>11}   which gates")
print("-" * 78)
for label, lines, total, which in rows:
    print(f"{label:<22}{lines:>7}{total:>11}   {which[:38]}")

fwd_lines, fwd_gates = rows[0][1] + rows[1][1], rows[0][2] + rows[1][2]
bwd_lines, bwd_gates = rows[2][1] + rows[3][1], rows[2][2] + rows[3][2]
print(f"\nforward : {fwd_lines:>4} lines, {fwd_gates:>2} gate references")
print(f"backward: {bwd_lines:>4} lines, {bwd_gates:>2} gate references")

assert bwd_gates == 0, "the backward was supposed to be gate-free"
print("\nThe backward is the LARGER half of the tutorial and mentions no architecture at all.")


### Interpret: the strip is empty

![The forward is guarded by 33 gates; the backward by none](../course/figures/fig_t06d_gates.svg)

The backward is the *bigger* half of the tutorial — 297 lines against the forward's 273 — and it
names no architecture at all. There is no HIP branch to delete, no descriptor to erase, no
`warp_specialize` to drop, no FP8 dtype to refuse.

Why? Because **the backward was written in the idiom 06c's §3 concluded was the right one all
along**: raw `tl.load` on hand-computed pointers.

```python
# _attn_bwd_dkdv — how it has always addressed memory
qT_ptrs = Q + offs_m[None, :] * stride_tok + offs_k[:, None] * stride_d
do_ptrs = DO + offs_m[:, None] * stride_tok + offs_k[None, :] * stride_d
```

06c had to *prove* that a tensor descriptor lowers to these same `cp.async` + `ldmatrix` loads on
Ada. The backward's authors never introduced the abstraction, so nothing had to be undone.

**Which leaves the interesting question.** A kernel with no `if` on your hardware can still depend on
it deeply — through the *resources* it asks for. Hold that thought until §10.


## 2. Ask your own GPU

This cell imports `torch`/`triton` if they exist and sets `HAS_TRITON`.

If they don't exist, it installs a **no-op shim** so every `@triton.jit` kernel below still *defines*
without error (the decorator becomes the identity function). Nothing runs on a GPU, but the notebook
stays readable, the exercises still run, and `course/review-notebook.py` can execute every cell under
plain Python.

We also read the two shared-memory numbers that §10 and §11 turn on. Note them for your card.


In [ ]:
try:
    import torch
    import triton
    import triton.language as tl

    HAS_TRITON = torch.cuda.is_available()
except ImportError:
    HAS_TRITON = False

    class _Shim:
        """No-op stand-in: @triton.jit -> identity, triton.Config(...) -> harmless object."""

        def __getattr__(self, _name):
            def _deco(*args, **kwargs):
                if len(args) == 1 and not kwargs and callable(args[0]):
                    return args[0]
                return lambda f: f
            return _deco

    class _ShimTorch(_Shim):
        """Same, plus a real subclassable Function: §7's `class _attention(torch.autograd.Function)`
        evaluates its base class at definition time, even though nothing will ever run it."""

        class autograd:
            class Function:
                @staticmethod
                def apply(*_args, **_kwargs):
                    raise RuntimeError("torch is not installed; these kernels cannot run")

    triton = tl = _Shim()
    if "torch" not in globals():          # torch can exist even when triton does not
        torch = _ShimTorch()

# Recorded on the reference card (RTX 4080 SUPER, sm_89) -- used when there is no GPU.
SMEM_PER_SM, SMEM_PER_BLOCK = 102400, 101376

if HAS_TRITON:
    props = torch.cuda.get_device_properties(0)
    major, minor = torch.cuda.get_device_capability()
    SMEM_PER_SM = props.shared_memory_per_multiprocessor
    SMEM_PER_BLOCK = props.shared_memory_per_block_optin
    print(f"device      : {props.name}")
    print(f"capability  : {major}.{minor}")
    print(f"triton      : {triton.__version__}")
    print(f"torch       : {torch.__version__}")
else:
    print("No torch+CUDA here. Kernels will DEFINE (via the shim) but not RUN.")
    print("Exercises are pure NumPy and run fine. Cells print the recorded numbers.\n")
    print("device      : NVIDIA GeForce RTX 4080 SUPER   (recorded)")
    print("capability  : 8.9")

print(f"\nshared memory / SM          : {SMEM_PER_SM:>7} bytes  ({SMEM_PER_SM / 1024:.0f} KB)")
print(f"shared memory / thread block: {SMEM_PER_BLOCK:>7} bytes  ({SMEM_PER_BLOCK / 1024:.0f} KB)")
print(f"\n-> Two thread blocks share an SM only while each uses <= {SMEM_PER_SM // 2} bytes.")
print("   Remember that number. It decides §11.")


## 3. The forward, carried over from 06c

The backward needs two things the forward produced: the output `O`, and the per-row logsumexp `M`
that lets it **recompute** `P` instead of storing it (06b §4).

Below is 06c's stripped forward, **unchanged**. Read it only if you skipped 06c; otherwise go
straight to §4 — nothing here is new, and nothing here is what this chapter is about.


In [ ]:
# ---- carried over from 06c, verbatim. Nothing to learn here; §4 is where 06d starts. ----
configs = [
    triton.Config({"BLOCK_M": BLOCK_M, "BLOCK_N": BLOCK_N}, num_stages=s, num_warps=w)
    for BLOCK_M in [64, 128] for BLOCK_N in [32, 64] for s in [2, 3, 4] for w in [4, 8]
]


def prune_invalid_configs(configs, named_args, **kwargs):
    """SHAPE guard, not an arch guard: BLOCK_M must fit N_CTX, and cover BLOCK_N when causal."""
    N_CTX, STAGE = kwargs["N_CTX"], kwargs["STAGE"]
    return [c for c in configs
            if c.kwargs["BLOCK_M"] <= N_CTX and (c.kwargs["BLOCK_M"] >= c.kwargs["BLOCK_N"] or STAGE == 1)]


@triton.jit
def _fwd_inner(acc, l_i, m_i, q, K_ptr, V_ptr, stride_kn, stride_vn, start_m, qk_scale,
               BLOCK_M: tl.constexpr, HEAD_DIM: tl.constexpr, BLOCK_N: tl.constexpr,
               STAGE: tl.constexpr, offs_m: tl.constexpr, offs_n: tl.constexpr, N_CTX: tl.constexpr):
    if STAGE == 1:
        lo, hi = 0, start_m * BLOCK_M
    elif STAGE == 2:
        lo, hi = start_m * BLOCK_M, (start_m + 1) * BLOCK_M
        lo = tl.multiple_of(lo, BLOCK_M)
    else:
        lo, hi = 0, N_CTX
    K_ptr += lo * stride_kn
    V_ptr += lo * stride_vn
    for start_n in tl.range(lo, hi, BLOCK_N):
        start_n = tl.multiple_of(start_n, BLOCK_N)
        k = tl.load(K_ptr)
        qk = tl.dot(q, k)
        if STAGE == 2:
            mask = offs_m[:, None] >= (start_n + offs_n[None, :])
            qk = qk * qk_scale + tl.where(mask, 0, -1.0e6)
            m_ij = tl.maximum(m_i, tl.max(qk, 1))
            qk -= m_ij[:, None]
        else:
            m_ij = tl.maximum(m_i, tl.max(qk, 1) * qk_scale)
            qk = qk * qk_scale - m_ij[:, None]
        p = tl.math.exp2(qk)
        alpha = tl.math.exp2(m_i - m_ij)
        l_ij = tl.sum(p, 1)
        acc = acc * alpha[:, None]
        v = tl.load(V_ptr)
        acc = tl.dot(p.to(tl.float16), v, acc)
        l_i = l_i * alpha + l_ij
        m_i = m_ij
        K_ptr += BLOCK_N * stride_kn
        V_ptr += BLOCK_N * stride_vn
    return acc, l_i, m_i


@triton.autotune(configs=configs, key=["N_CTX", "HEAD_DIM"],
                 prune_configs_by={"early_config_prune": prune_invalid_configs})
@triton.jit
def _attn_fwd(Q, K, V, sm_scale, M, Out,
              stride_qz, stride_qh, stride_qm, stride_qk,
              stride_kz, stride_kh, stride_kn, stride_kk,
              stride_vz, stride_vh, stride_vn, stride_vk,
              stride_oz, stride_oh, stride_om, stride_on,
              Z, H, N_CTX, HEAD_DIM: tl.constexpr, BLOCK_M: tl.constexpr,
              BLOCK_N: tl.constexpr, STAGE: tl.constexpr):
    tl.static_assert(BLOCK_N <= HEAD_DIM)
    start_m, off_hz = tl.program_id(0), tl.program_id(1)
    off_z, off_h = off_hz // H, off_hz % H
    q_base = Q + off_z.to(tl.int64) * stride_qz + off_h.to(tl.int64) * stride_qh
    k_base = K + off_z.to(tl.int64) * stride_kz + off_h.to(tl.int64) * stride_kh
    v_base = V + off_z.to(tl.int64) * stride_vz + off_h.to(tl.int64) * stride_vh
    o_base = Out + off_z.to(tl.int64) * stride_oz + off_h.to(tl.int64) * stride_oh
    offs_m = start_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = tl.arange(0, BLOCK_N)
    offs_d = tl.arange(0, HEAD_DIM)
    m_i = tl.zeros([BLOCK_M], dtype=tl.float32) - float("inf")
    l_i = tl.zeros([BLOCK_M], dtype=tl.float32) + 1.0
    acc = tl.zeros([BLOCK_M, HEAD_DIM], dtype=tl.float32)
    qk_scale = sm_scale * 1.44269504
    q = tl.load(q_base + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qk)
    K_ptr = k_base + offs_n[None, :] * stride_kn + offs_d[:, None] * stride_kk
    V_ptr = v_base + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vk
    if STAGE & 1:
        acc, l_i, m_i = _fwd_inner(acc, l_i, m_i, q, K_ptr, V_ptr, stride_kn, stride_vn, start_m,
                                   qk_scale, BLOCK_M, HEAD_DIM, BLOCK_N, 4 - STAGE, offs_m, offs_n, N_CTX)
    if STAGE & 2:
        acc, l_i, m_i = _fwd_inner(acc, l_i, m_i, q, K_ptr, V_ptr, stride_kn, stride_vn, start_m,
                                   qk_scale, BLOCK_M, HEAD_DIM, BLOCK_N, 2, offs_m, offs_n, N_CTX)
    m_i += tl.math.log2(l_i)                     # the logsumexp the backward will consume
    acc = acc / l_i[:, None]
    tl.store(M + off_hz * N_CTX + offs_m, m_i)
    tl.store(o_base + offs_m[:, None] * stride_om + offs_d[None, :] * stride_on, acc.to(tl.float16))


print("forward carried over from 06c:", len(configs), "autotune configs")


## 4. `_attn_bwd_preprocess` — the first real launch

06b showed that the awkward `rowsum(P ∘ dP)` collapses to `D_i = dO_i · O_i`, so the backward never
needs `P` for that term. The kernel that computes it is four lines long.

Here it is as real Triton. Note what it *doesn't* have: no strides (it assumes `O` and `dO` are
contiguous `(Z, H, N_CTX, HEAD_DIM)`), no masking (it assumes `N_CTX % BLOCK_M == 0`), and no
architecture anything.


In [ ]:
@triton.jit
def _attn_bwd_preprocess(O, DO, Delta, Z, H, N_CTX,
                         BLOCK_M: tl.constexpr, HEAD_DIM: tl.constexpr):
    off_m = tl.program_id(0) * BLOCK_M + tl.arange(0, BLOCK_M)
    off_hz = tl.program_id(1)
    off_n = tl.arange(0, HEAD_DIM)
    o = tl.load(O + off_hz * HEAD_DIM * N_CTX + off_m[:, None] * HEAD_DIM + off_n[None, :])
    do = tl.load(DO + off_hz * HEAD_DIM * N_CTX + off_m[:, None] * HEAD_DIM + off_n[None, :]).to(tl.float32)
    delta = tl.sum(o * do, axis=1)                       # D_i = rowsum(O ∘ dO)
    tl.store(Delta + off_hz * N_CTX + off_m, delta)


print("_attn_bwd_preprocess defined")


### Launch it, and check `D` against PyTorch

`Delta` is the one place the backward touches `O` at all. If this is wrong, every `dS` is wrong.
Launching it alone is the cheapest possible confidence.

**Predict before you run:** `Delta` is `float32` while `O` and `dO` are `float16`. Where does the
rounding happen, and does it matter?


In [ ]:
if HAS_TRITON:
    torch.manual_seed(0)
    Z_, H_, N_, D_ = 2, 4, 512, 64
    o = torch.randn((Z_, H_, N_, D_), dtype=torch.float16, device="cuda")
    do = torch.randn_like(o)
    delta = torch.empty((Z_, H_, N_), dtype=torch.float32, device="cuda")

    PRE_BLOCK = 128
    _attn_bwd_preprocess[(N_ // PRE_BLOCK, Z_ * H_)](o, do, delta, Z_, H_, N_,
                                                     BLOCK_M=PRE_BLOCK, HEAD_DIM=D_)

    ref = (o.float() * do.float()).sum(-1)               # the same reduction, in torch
    err = (delta - ref).abs().max().item()
    print(f"max |Delta_triton - Delta_torch| = {err:.3e}")
    assert err < 1e-3, "preprocess disagrees with torch"
    print("PASS — D = rowsum(O ∘ dO), computed on the GPU.")
else:
    print("Recorded on RTX 4080 SUPER: max |Delta_triton - Delta_torch| = 0.000e+00")
    print("PASS — the reduction upcasts before the multiply, so it matches torch's fp32 reduction.")


### Interpret

`do` is cast with `.to(tl.float32)` **before** the elementwise product, and `tl.sum` accumulates in
fp32. So the only fp16 rounding is in the *inputs* — which torch's reference shares. The two agree to
the bit.

That is not a detail. `D` is subtracted from `dP` inside `dS = P ∘ (dP − D)`, where `dP` and `D` are
similar in magnitude: a catastrophic-cancellation site. Computing it in fp16 would be a real bug, and
it is why the kernel bothers to upcast.


## 5. `_attn_bwd_dkdv` and `_attn_bwd_dq`

The two inner loops, verbatim from the tutorial. 06b walked both line by line; here they are as
executable code. Three things to notice on the way past:

- **`qT_ptrs` is built transposed** — `offs_m[None, :] * stride_tok + offs_k[:, None] * stride_d`
  yields a `(HEAD_DIM, BLOCK_M1)` tile, so `tl.dot(k, qT)` needs no `.T`. Same trick as 06c's `K_ptr`.
- **`pT = exp2(qkT - m)` is the recompute** — `m` is the forward's logsumexp, so this *is* the softmax
  probability, rebuilt from a vector instead of read from an `(N, N)` matrix.
- **`MASK` is a `tl.constexpr`** — the masked diagonal and the off-diagonal bulk compile to two
  different kernels. No branch is taken at runtime.


In [ ]:
@triton.jit
def _attn_bwd_dkdv(dk, dv, Q, k, v, sm_scale, DO, M, D, stride_tok, stride_d,
                   H, N_CTX, BLOCK_M1: tl.constexpr, BLOCK_N1: tl.constexpr, HEAD_DIM: tl.constexpr,
                   start_n, start_m, num_steps, MASK: tl.constexpr):
    offs_m = start_m + tl.arange(0, BLOCK_M1)
    offs_n = start_n + tl.arange(0, BLOCK_N1)
    offs_k = tl.arange(0, HEAD_DIM)
    qT_ptrs = Q + offs_m[None, :] * stride_tok + offs_k[:, None] * stride_d   # (HEAD_DIM, BLOCK_M1)
    do_ptrs = DO + offs_m[:, None] * stride_tok + offs_k[None, :] * stride_d
    tl.static_assert(BLOCK_N1 % BLOCK_M1 == 0)
    curr_m = start_m
    step_m = BLOCK_M1
    for blk_idx in range(num_steps):
        qT = tl.load(qT_ptrs)
        offs_m = curr_m + tl.arange(0, BLOCK_M1)
        m = tl.load(M + offs_m)                     # forward's logsumexp -> recompute P
        qkT = tl.dot(k, qT)
        pT = tl.math.exp2(qkT - m[None, :])
        if MASK:
            mask = (offs_m[None, :] >= offs_n[:, None])
            pT = tl.where(mask, pT, 0.0)
        do = tl.load(do_ptrs)
        ppT = pT.to(tl.float16)
        dv += tl.dot(ppT, do)                       # dV_j += P_ij^T dO_i
        Di = tl.load(D + offs_m)
        dpT = tl.dot(v, tl.trans(do)).to(tl.float32)
        dsT = pT * (dpT - Di[None, :])              # dS = P ∘ (dP - D)
        dsT = dsT.to(tl.float16)
        dk += tl.dot(dsT, tl.trans(qT))             # dK_j += dS_ij^T Q_i
        curr_m += step_m
        qT_ptrs += step_m * stride_tok
        do_ptrs += step_m * stride_tok
    return dk, dv


@triton.jit
def _attn_bwd_dq(dq, q, K, V, do, m, D, stride_tok, stride_d, H, N_CTX,
                 BLOCK_M2: tl.constexpr, BLOCK_N2: tl.constexpr, HEAD_DIM: tl.constexpr,
                 start_m, start_n, num_steps, MASK: tl.constexpr):
    offs_m = start_m + tl.arange(0, BLOCK_M2)
    offs_n = start_n + tl.arange(0, BLOCK_N2)
    offs_k = tl.arange(0, HEAD_DIM)
    kT_ptrs = K + offs_n[None, :] * stride_tok + offs_k[:, None] * stride_d
    vT_ptrs = V + offs_n[None, :] * stride_tok + offs_k[:, None] * stride_d
    Di = tl.load(D + offs_m)
    tl.static_assert(BLOCK_M2 % BLOCK_N2 == 0)
    curr_n = start_n
    step_n = BLOCK_N2
    for blk_idx in range(num_steps):
        kT = tl.load(kT_ptrs)
        vT = tl.load(vT_ptrs)
        qk = tl.dot(q, kT)
        p = tl.math.exp2(qk - m)
        if MASK:
            offs_n = curr_n + tl.arange(0, BLOCK_N2)
            mask = (offs_m[:, None] >= offs_n[None, :])
            p = tl.where(mask, p, 0.0)
        dp = tl.dot(do, vT).to(tl.float32)
        ds = p * (dp - Di[:, None])
        ds = ds.to(tl.float16)
        dq += tl.dot(ds, tl.trans(kT))              # dQ_i += dS_ij K_j
        curr_n += step_n
        kT_ptrs += step_n * stride_tok
        vT_ptrs += step_n * stride_tok
    return dq


print("_attn_bwd_dkdv and _attn_bwd_dq defined")


## 6. `_attn_bwd` — one program, two axes

Here is the design decision 06b explained and this notebook can *measure*. A single program does two
jobs:

1. it owns **key block `pid`**, and sweeps *down* the query axis accumulating `dK_pid`, `dV_pid`;
2. then it owns **query block `pid`**, and sweeps *across* the key axis accumulating `dQ_pid`.

Why fuse two loops with opposite reduction axes into one kernel? Because **causally, their work is
exactly complementary.**

Key block `pid` is attended to by every query *after* it — so late key blocks do little `dK/dV` work.
Query block `pid` attends to every key *before* it — so late query blocks do a lot of `dQ` work. Give
both jobs to the same program and the two triangles fill each other's gaps.

Let's count the inner-loop steps and see.


In [ ]:
BLOCK_M1, BLOCK_N1, BLOCK_M2, BLOCK_N2 = 32, 128, 128, 32   # upstream's constants
BLK_SLICE_FACTOR = 2
MASK_BLOCK_M1 = BLOCK_M1 // BLK_SLICE_FACTOR                # finer tiles on the masked diagonal
MASK_BLOCK_N2 = BLOCK_N2 // BLK_SLICE_FACTOR


def bwd_steps(pid, N_CTX, causal):
    """Inner-loop steps program `pid` runs, exactly as _attn_bwd computes them."""
    if not causal:
        return {"dkdv_masked": 0, "dkdv_bulk": N_CTX // BLOCK_M1,
                "dq_masked": 0, "dq_bulk": N_CTX // BLOCK_N2}
    start_n = pid * BLOCK_N1
    dkdv_masked = BLOCK_N1 // MASK_BLOCK_M1                 # the diagonal, in fine tiles
    start_m = start_n + dkdv_masked * MASK_BLOCK_M1
    dkdv_bulk = (N_CTX - start_m) // BLOCK_M1               # queries strictly after this key block

    start_m2 = pid * BLOCK_M2
    dq_masked = BLOCK_M2 // MASK_BLOCK_N2
    end_n = start_m2 + BLOCK_M2 - dq_masked * MASK_BLOCK_N2
    dq_bulk = end_n // BLOCK_N2                             # keys strictly before this query block
    return {"dkdv_masked": dkdv_masked, "dkdv_bulk": dkdv_bulk,
            "dq_masked": dq_masked, "dq_bulk": dq_bulk}


N_CTX_ = 1024
print(f"causal, N_CTX={N_CTX_}, {N_CTX_ // BLOCK_N1} programs\n")
print(f"{'pid':>4} {'dkdv masked':>12} {'dkdv bulk':>10} {'dq masked':>10} {'dq bulk':>8} {'TOTAL':>7}")
totals = []
for pid in range(N_CTX_ // BLOCK_N1):
    s = bwd_steps(pid, N_CTX_, causal=True)
    t = sum(s.values())
    totals.append(t)
    print(f"{pid:>4} {s['dkdv_masked']:>12} {s['dkdv_bulk']:>10} {s['dq_masked']:>10} "
          f"{s['dq_bulk']:>8} {t:>7}")

assert len(set(totals)) == 1, f"programs are NOT balanced: {totals}"
print(f"\nEvery program runs exactly {totals[0]} steps. The causal triangle is perfectly balanced.")


### Interpret: the imbalance cancels

![dQ's work grows with pid exactly as dK/dV's shrinks](../course/figures/fig_t06d_balance.svg)

`dkdv_bulk` falls by 4 per program; `dq_bulk` rises by 4. Their sum is constant, so **every program
does the same amount of work** — 44 steps at `N_CTX=1024`, and `16 + (N_CTX − 128)/32` in general.

Compare the *forward*, where 06c's Exercise C found the opposite: causal program `start_m=0` runs 2
inner steps and `start_m=7` runs 16. FlashAttention-2's forward simply accepts that imbalance. Its
backward gets balance for free, by pairing each key block with the query block of the same index.

That is the real reason `_attn_bwd` is one kernel and not two. Now the driver itself.


In [ ]:
@triton.jit
def _attn_bwd(Q, K, V, sm_scale, DO, DQ, DK, DV, M, D,
              stride_z, stride_h, stride_tok, stride_d, H, N_CTX,
              BLOCK_M1: tl.constexpr, BLOCK_N1: tl.constexpr,
              BLOCK_M2: tl.constexpr, BLOCK_N2: tl.constexpr,
              BLK_SLICE_FACTOR: tl.constexpr, HEAD_DIM: tl.constexpr, CAUSAL: tl.constexpr,
              SCALE_DK: tl.constexpr, SCALE_DQ: tl.constexpr):   # <- the two epilogue flags, for §9
    LN2: tl.constexpr = 0.6931471824645996

    bhid = tl.program_id(2)
    off_chz = (bhid * N_CTX).to(tl.int64)
    adj = (stride_h * (bhid % H) + stride_z * (bhid // H)).to(tl.int64)
    pid = tl.program_id(0)
    Q += adj
    K += adj
    V += adj
    DO += adj
    DQ += adj
    DK += adj
    DV += adj
    M += off_chz
    D += off_chz
    offs_k = tl.arange(0, HEAD_DIM)

    # ---------------- this program owns KEY block `pid`: accumulate dK, dV
    start_n = pid * BLOCK_N1
    start_m = 0
    MASK_BLOCK_M1: tl.constexpr = BLOCK_M1 // BLK_SLICE_FACTOR
    offs_n = start_n + tl.arange(0, BLOCK_N1)
    dv = tl.zeros([BLOCK_N1, HEAD_DIM], dtype=tl.float32)
    dk = tl.zeros([BLOCK_N1, HEAD_DIM], dtype=tl.float32)
    k = tl.load(K + offs_n[:, None] * stride_tok + offs_k[None, :] * stride_d)
    v = tl.load(V + offs_n[:, None] * stride_tok + offs_k[None, :] * stride_d)

    if CAUSAL:                                       # the masked diagonal, in finer tiles
        start_m = start_n
        num_steps = BLOCK_N1 // MASK_BLOCK_M1
        dk, dv = _attn_bwd_dkdv(dk, dv, Q, k, v, sm_scale, DO, M, D, stride_tok, stride_d,
                                H, N_CTX, MASK_BLOCK_M1, BLOCK_N1, HEAD_DIM,
                                start_n, start_m, num_steps, MASK=True)
        start_m += num_steps * MASK_BLOCK_M1

    num_steps = (N_CTX - start_m) // BLOCK_M1        # the unmasked bulk
    dk, dv = _attn_bwd_dkdv(dk, dv, Q, k, v, sm_scale, DO, M, D, stride_tok, stride_d,
                            H, N_CTX, BLOCK_M1, BLOCK_N1, HEAD_DIM,
                            start_n, start_m, num_steps, MASK=False)

    tl.store(DV + offs_n[:, None] * stride_tok + offs_k[None, :] * stride_d, dv)
    if SCALE_DK:
        dk *= sm_scale                               # §9: dK multiplied Q, never the prescaled arg_k
    tl.store(DK + offs_n[:, None] * stride_tok + offs_k[None, :] * stride_d, dk)

    # ---------------- the SAME program now owns QUERY block `pid`: accumulate dQ
    start_m = pid * BLOCK_M2
    start_n = 0
    num_steps = N_CTX // BLOCK_N2
    MASK_BLOCK_N2: tl.constexpr = BLOCK_N2 // BLK_SLICE_FACTOR
    offs_m = start_m + tl.arange(0, BLOCK_M2)
    q = tl.load(Q + offs_m[:, None] * stride_tok + offs_k[None, :] * stride_d)
    dq = tl.zeros([BLOCK_M2, HEAD_DIM], dtype=tl.float32)
    do = tl.load(DO + offs_m[:, None] * stride_tok + offs_k[None, :] * stride_d)
    m = tl.load(M + offs_m)
    m = m[:, None]

    if CAUSAL:
        end_n = start_m + BLOCK_M2
        num_steps = BLOCK_M2 // MASK_BLOCK_N2
        dq = _attn_bwd_dq(dq, q, K, V, do, m, D, stride_tok, stride_d, H, N_CTX,
                          BLOCK_M2, MASK_BLOCK_N2, HEAD_DIM,
                          start_m, end_n - num_steps * MASK_BLOCK_N2, num_steps, MASK=True)
        end_n -= num_steps * MASK_BLOCK_N2
        num_steps = end_n // BLOCK_N2
        start_n = end_n - num_steps * BLOCK_N2

    dq = _attn_bwd_dq(dq, q, K, V, do, m, D, stride_tok, stride_d, H, N_CTX,
                      BLOCK_M2, BLOCK_N2, HEAD_DIM, start_m, start_n, num_steps, MASK=False)
    if SCALE_DQ:
        dq *= LN2                                    # §9: dQ multiplied arg_k, so it carries log2(e)
    tl.store(DQ + offs_m[:, None] * stride_tok + offs_k[None, :] * stride_d, dq)


print("_attn_bwd defined — one kernel, both axes")


## 7. Wiring it into `torch.autograd`

Three kernels, one `Function`. `forward` runs `_attn_fwd` and saves `(q, k, v, o, M)`; `backward`
runs `_attn_bwd_preprocess` then `_attn_bwd`.

Two details worth your attention:

- **`arg_k = k * (sm_scale * RCP_LN2)`.** The scale and the `log2(e)` change-of-base are folded into
  `K` once, on the host, so the inner loops can use `exp2` and never multiply by `sm_scale`. The two
  epilogue rescales (`dk *= sm_scale`, `dq *= LN2`) put the units back. 06b derived this; §9 breaks it
  on purpose.
- **`num_stages=5`** is what upstream hardcodes. We keep it, for now. It will not survive §10.

> **Gotcha:** `torch.autograd.Function.apply` accepts **no keyword arguments**. Every extra flag below
> must be passed positionally — a real trap when you add a debug switch to a kernel wrapper.


In [ ]:
class _attention(torch.autograd.Function):

    @staticmethod
    def forward(ctx, q, k, v, causal, sm_scale,
                prescale_k=True, scale_dk=True, scale_dq=True, num_stages=5):
        HEAD_DIM = q.shape[-1]
        assert HEAD_DIM in {16, 32, 64, 128, 256}
        assert q.shape[2] % 128 == 0, "no boundary masking: N_CTX must be a multiple of 128"
        o = torch.empty_like(q)
        M = torch.empty((q.shape[0], q.shape[1], q.shape[2]), device=q.device, dtype=torch.float32)
        stage = 3 if causal else 1
        grid = lambda META: (triton.cdiv(q.shape[2], META["BLOCK_M"]), q.shape[0] * q.shape[1], 1)
        _attn_fwd[grid](q, k, v, sm_scale, M, o,
                        *q.stride(), *k.stride(), *v.stride(), *o.stride(),
                        q.shape[0], q.shape[1], N_CTX=q.shape[2], HEAD_DIM=HEAD_DIM, STAGE=stage)
        ctx.save_for_backward(q, k, v, o, M)
        ctx.sm_scale, ctx.HEAD_DIM, ctx.causal = sm_scale, HEAD_DIM, causal
        ctx.prescale_k, ctx.scale_dk, ctx.scale_dq = prescale_k, scale_dk, scale_dq
        ctx.num_stages = num_stages
        return o

    @staticmethod
    def backward(ctx, do):
        q, k, v, o, M = ctx.saved_tensors
        do = do.contiguous()
        dq, dk, dv = torch.empty_like(q), torch.empty_like(k), torch.empty_like(v)
        BATCH, N_HEAD, N_CTX = q.shape[:3]
        PRE_BLOCK = 128
        RCP_LN2 = 1.4426950408889634                 # = 1 / ln(2)

        # Fold sm_scale AND the change of base into K, once, on the host.
        arg_k = k * (ctx.sm_scale * RCP_LN2) if ctx.prescale_k else k

        assert N_CTX % PRE_BLOCK == 0
        delta = torch.empty_like(M)
        _attn_bwd_preprocess[(N_CTX // PRE_BLOCK, BATCH * N_HEAD)](
            o, do, delta, BATCH, N_HEAD, N_CTX, BLOCK_M=PRE_BLOCK, HEAD_DIM=ctx.HEAD_DIM)

        _attn_bwd[(N_CTX // BLOCK_N1, 1, BATCH * N_HEAD)](
            q, arg_k, v, ctx.sm_scale, do, dq, dk, dv, M, delta,
            q.stride(0), q.stride(1), q.stride(2), q.stride(3), N_HEAD, N_CTX,
            BLOCK_M1=BLOCK_M1, BLOCK_N1=BLOCK_N1, BLOCK_M2=BLOCK_M2, BLOCK_N2=BLOCK_N2,
            BLK_SLICE_FACTOR=BLK_SLICE_FACTOR, HEAD_DIM=ctx.HEAD_DIM,
            num_warps=4, num_stages=ctx.num_stages, CAUSAL=ctx.causal,
            SCALE_DK=ctx.scale_dk, SCALE_DQ=ctx.scale_dq)

        return dq, dk, dv, None, None, None, None, None, None


attention = _attention.apply


def reference_attention(q, k, v, causal, sm_scale):
    """Textbook attention; softmax in fp32. Its autograd graph is our ground truth."""
    N = q.shape[2]
    p = torch.matmul(q, k.transpose(2, 3)) * sm_scale
    if causal:
        mask = torch.tril(torch.ones((N, N), device=q.device))
        p = p.masked_fill(mask == 0, float("-inf"))
    p = torch.softmax(p.float(), dim=-1).half()
    return torch.matmul(p, v)


print("_attention wired: forward -> _attn_fwd; backward -> _attn_bwd_preprocess + _attn_bwd")


## 8. Do the gradients agree with PyTorch?

The only acceptable evidence. We build `q, k, v` with `requires_grad_()`, run both our kernel and the
reference, backprop the same `dO` through each, and compare all three gradients.

We stay at **`HEAD_DIM=64`** here. (Why not 128? §10.)


In [ ]:
def make_qkv(Z, H, N, D, seed=20):
    torch.manual_seed(seed)
    return [torch.empty((Z, H, N, D), dtype=torch.float16, device="cuda")
            .normal_(mean=0.0, std=0.5).requires_grad_() for _ in range(3)]


def grads_of(fn, q, k, v, causal, sm_scale, dout, *extra):
    """*extra is positional: Function.apply takes no kwargs."""
    for t in (q, k, v):
        t.grad = None
    out = fn(q, k, v, causal, sm_scale, *extra)
    out.backward(dout, retain_graph=True)
    return out, q.grad.clone(), k.grad.clone(), v.grad.clone()


if HAS_TRITON:
    print(f"{'Z':>2} {'H':>3} {'N_CTX':>6} {'D':>4} {'causal':>7} "
          f"{'max|dO err|':>12} {'max|dQ err|':>12} {'max|dK err|':>12} {'max|dV err|':>12}  status")
    worst = 0.0
    for (Z_, H_, N_, D_) in [(1, 2, 1024, 64), (2, 4, 512, 64), (4, 8, 2048, 64)]:
        for causal in (False, True):
            q, k, v = make_qkv(Z_, H_, N_, D_)
            dout = torch.randn_like(q)
            got = grads_of(attention, q, k, v, causal, 0.5, dout)
            want = grads_of(reference_attention, q, k, v, causal, 0.5, dout)
            errs = [(a - b).abs().max().item() for a, b in zip(got, want)]
            worst = max(worst, max(errs))
            print(f"{Z_:>2} {H_:>3} {N_:>6} {D_:>4} {str(causal):>7} " +
                  " ".join(f"{e:>12.3e}" for e in errs) + f"  {'OK' if max(errs) < 1e-2 else 'FAIL'}")
            del q, k, v, dout
            torch.cuda.empty_cache()
    assert worst < 1e-2, f"gradients diverged: {worst}"
    print(f"\nPASS — worst error {worst:.3e} over all six cases (upstream's test tolerance is 1e-2).")
else:
    print("Recorded on RTX 4080 SUPER, HEAD_DIM=64:\n")
    print(" Z   H  N_CTX    D  causal   max|dO err|  max|dQ err|  max|dK err|  max|dV err|")
    print(" 1   2   1024   64   False     2.441e-04    4.883e-04    4.883e-04    7.324e-04")
    print(" 1   2   1024   64    True     4.883e-04    1.953e-03    1.953e-03    1.953e-03")
    print(" 2   4    512   64   False     3.662e-04    9.766e-04    6.104e-04    7.324e-04")
    print(" 2   4    512   64    True     9.766e-04    1.953e-03    1.953e-03    1.953e-03")
    print(" 4   8   2048   64   False     1.831e-04    4.883e-04    5.341e-04    4.883e-04")
    print(" 4   8   2048   64    True     9.766e-04    1.953e-03    1.953e-03    3.906e-03")
    print("\nPASS — worst error 3.906e-03 (tolerance 1e-2).")


### Interpret

Errors sit around `10⁻³`, and the causal cases are consistently worse than the non-causal ones. That
is expected: causal rows have *fewer* terms in the softmax denominator, so each surviving `P_ij` is
larger, and fp16's `2⁻¹¹` relative spacing bites harder on the `dS = P ∘ (dP − D)` cancellation.

`10⁻³` on values of order 1 is roughly **fp16 resolution**. The gradients are as right as fp16 allows.


## 9. Break it on purpose — the three scalings

06b derived why the backward needs exactly three scaling steps. Let's find out what each one is
*worth*, by deleting it and watching the gradients.

| Step | Where | What it does |
|---|---|---|
| `arg_k = k * (sm_scale * RCP_LN2)` | host, before launch | folds `sm_scale` and `log2(e)` into `K` |
| `dk *= sm_scale` | `_attn_bwd` epilogue | `dK` multiplied `Q`, never the prescaled `arg_k` |
| `dq *= LN2` | `_attn_bwd` epilogue | `dQ` multiplied `arg_k`, so it carries a spurious `log2(e)` |

The kernel already carries `SCALE_DK` and `SCALE_DQ` as `constexpr` flags, and the wrapper carries
`prescale_k`. Flip one at a time.

**Predict before you run.** For each broken variant: *which* gradients go wrong, and by what factor?
Write your three answers down before executing.


In [ ]:
LN2, RCP_LN2, SM_SCALE = 0.6931471824645996, 1.4426950408889634, 0.5

VARIANTS = [                            # (prescale_k, scale_dk, scale_dq)
    ("correct (all three)", (True, True, True)),
    ("drop arg_k prescale", (False, True, True)),
    ("drop  dq *= LN2", (True, True, False)),
    ("drop  dk *= sm_scale", (True, False, True)),
]

if HAS_TRITON:
    q, k, v = make_qkv(1, 2, 512, 64)
    dout = torch.randn_like(q)
    _, dq_ref, dk_ref, dv_ref = grads_of(reference_attention, q, k, v, False, SM_SCALE, dout)
    _, dq_ok, dk_ok, dv_ok = grads_of(attention, q, k, v, False, SM_SCALE, dout)

    def median_ratio(bad, good):
        """Ratio on the entries big enough for the ratio to be meaningful."""
        big = good.abs() > good.abs().max() * 0.5
        return (bad[big].float() / good[big].float()).median().item()

    print(f"{'variant':<22}{'max|dQ err|':>13}{'max|dK err|':>13}{'max|dV err|':>13}"
          f"{'dQ ratio':>11}{'dK ratio':>11}")
    print("-" * 83)
    for name, flags in VARIANTS:
        _, dq, dk, dv = grads_of(attention, q, k, v, False, SM_SCALE, dout, *flags)
        errs = [(a - b).abs().max().item() for a, b in [(dq, dq_ref), (dk, dk_ref), (dv, dv_ref)]]
        print(f"{name:<22}" + "".join(f"{e:>13.3e}" for e in errs) +
              f"{median_ratio(dq, dq_ok):>11.4f}{median_ratio(dk, dk_ok):>11.4f}")
    del q, k, v, dout
    torch.cuda.empty_cache()
else:
    print("Recorded on RTX 4080 SUPER (Z=1 H=2 N_CTX=512 HEAD_DIM=64, sm_scale=0.5, causal=False):\n")
    print("variant                 max|dQ err|  max|dK err|  max|dV err|   dQ ratio   dK ratio")
    print("-" * 83)
    print("correct (all three)       4.883e-04    4.883e-04    4.883e-04     1.0000     1.0000")
    print("drop arg_k prescale       2.447e+00    2.314e+00    1.618e+00     3.7585     3.4710")
    print("drop  dq *= LN2           2.004e-01    4.883e-04    4.883e-04     1.4427     1.0000")
    print("drop  dk *= sm_scale      4.883e-04    6.206e-01    4.883e-04     1.0000     2.0000")

print(f"\n1 / ln(2)    = {RCP_LN2:.4f}   <- the dQ ratio when `dq *= LN2` is missing")
print(f"1 / sm_scale = {1 / SM_SCALE:.4f}   <- the dK ratio when `dk *= sm_scale` is missing")


### Interpret: two units bugs and one real bug

![Each epilogue scale corrects exactly one gradient](../course/figures/fig_t06d_ablation.svg)

Read the ratios, not the errors.

- **Drop `dq *= LN2`** → `dQ` is off by exactly **1.4427 = 1/ln 2**, and `dK`, `dV` are *untouched*.
- **Drop `dk *= sm_scale`** → `dK` is off by exactly **2.0 = 1/0.5 = 1/sm_scale**, others untouched.
- **Drop the `arg_k` prescale** → **all three** gradients are wrong, by ratios (3.76, 3.47) that are
  not constants at all — they depend on the data.

The distinction matters more than the arithmetic. The two epilogue rescales are **units bugs**: a
constant factor on one output, which a loss curve will happily absorb into the learning rate while
your model quietly trains on the wrong gradient. The prescale is a **semantic bug**: it changes the
argument of `exp2`, so `pT` is no longer the softmax, and *every* gradient is corrupted non-linearly.

> **Debugging heuristic.** A gradient wrong by a **clean constant** points at a missing epilogue
> scale. A gradient wrong by a **data-dependent factor** points into the softmax itself. And `dV`,
> which neither epilogue scale touches, is the tell-tale: if `dV` moved, the bug is upstream of the
> epilogue. Exercise D makes you do this diagnosis from the ratios alone.

A plain `assert torch.allclose(dq, dq_ref)` catches all three — but only the ratio tells you *which
line* to look at.


## 10. `HEAD_DIM=128`, and the wall

Everything so far ran at `HEAD_DIM=64`. The tutorial's own test suite also parametrizes
`HEAD_DIM=128`:

```python
@pytest.mark.parametrize("HEAD_DIM", [64, 128])
@pytest.mark.parametrize("mode", ["fwd", "bwd"])
def test_op(Z, H, N_CTX, HEAD_DIM, causal, warp_specialize, mode, provider, dtype=torch.float16):
```

The forward handles it fine — 06c benchmarked `HEAD_DIM=128` throughout. Let's ask the backward.


In [ ]:
if HAS_TRITON:
    q, k, v = make_qkv(1, 2, 256, 128)
    dout = torch.randn_like(q)
    out = attention(q, k, v, False, 0.5)          # forward at HEAD_DIM=128
    print(f"forward  at HEAD_DIM=128: OK, out.shape = {tuple(out.shape)}")
    try:
        out.backward(dout)
        print("backward at HEAD_DIM=128: OK")
    except triton.runtime.errors.OutOfResources as e:
        print(f"backward at HEAD_DIM=128: {type(e).__name__}")
        print(f"  {str(e).strip().splitlines()[0]}")
    del q, k, v, dout, out
    torch.cuda.empty_cache()
else:
    print("Recorded on RTX 4080 SUPER (sm_89):\n")
    print("forward  at HEAD_DIM=128: OK, out.shape = (1, 2, 256, 128)")
    print("backward at HEAD_DIM=128: OutOfResources")
    print("  out of resource: shared memory, Required: 109568, Hardware limit: 101376. "
          "Reducing block sizes or `num_stages` may help.")


### This is not our bug. It is the tutorial's.

Nothing above is a modification. `BLOCK_M1/N1/M2/N2 = 32/128/128/32`, `num_warps=4` and
`num_stages=5` are upstream's own hardcoded constants, and the kernels are verbatim. Run the genuine
`06-fused-attention.py` on an RTX 4080 and `test_op[…HEAD_DIM=128…mode=bwd]` raises the same
`OutOfResources`.

Do the arithmetic on the two numbers in that message:

$$\underbrace{109{,}568\ \text{bytes}}_{\text{required}} = 107\ \text{KB}
\qquad\qquad
\underbrace{101{,}376\ \text{bytes}}_{\text{hardware limit}} = 99\ \text{KB}$$

99 KB is not a Triton number. It is **the maximum shared memory per thread block on compute
capability 8.6 and 8.9** — every consumer Ampere card, every Ada card, and (per the Blackwell Tuning
Guide) every GeForce RTX 50xx.

![The backward's real Ampere/Ada constraint is a shared-memory budget](../course/figures/fig_t06d_smem.svg)

| cc | example | max smem / block | needs 107 KB? |
|---|---|---|---|
| 8.0 | A100 | 163 KB | fits |
| 8.6 | RTX 3090, A10 | **99 KB** | **will not launch** |
| 8.9 | RTX 4090, L40S | **99 KB** | **will not launch** |
| 9.0 | H100 | 227 KB | fits |
| 10.0 | B200 | 227 KB | fits |
| 12.0 | RTX 5090 | **99 KB** | **will not launch** |

So the tutorial's backward is not architecture-*neutral* after all. It was tuned on a card with
Hopper's 227 KB budget, and it silently assumes you have one. **"Ampere/Ada only" turns out to name a
real constraint on the backward — it just isn't written in the source.** It is written in the
compiler's shared-memory allocation, and you find it by launching.


## 11. The occupancy cliff, and a rule for `num_stages`

`num_stages` is the depth of Triton's **software pipeline**: how many loop iterations' worth of tiles
are prefetched into shared memory at once, so loads overlap with tensor-core math. More stages means
more overlap — and strictly more shared memory.

The obvious repair for §10 is to lower `num_stages` until it fits. But *which* value? The temptation
is "the largest that fits." That answer is wrong, and the reason is the most useful thing in this
notebook.

Here is the measured footprint of `_attn_bwd` (at upstream's block sizes) for each pipeline depth.


In [ ]:
# Measured with kernel.metadata.shared on sm_89, BLOCK_M1/N1/M2/N2=32/128/128/32, num_warps=4.
BWD_SMEM = {
    64:  {1: 40960, 2: 43264, 3: 47616, 4: 51968, 5: 56320, 6: 60672},
    128: {1: 81920, 2: 84224, 3: 92672, 4: 101120, 5: 109568, 6: 118016},
}


def blocks_per_sm(head_dim, num_stages, smem_per_sm):
    """How many thread blocks fit on one SM, as far as shared memory is concerned."""
    return smem_per_sm // BWD_SMEM[head_dim][num_stages]


print(f"shared memory per SM = {SMEM_PER_SM} B;  per thread block = {SMEM_PER_BLOCK} B\n")
for head_dim in (64, 128):
    print(f"HEAD_DIM = {head_dim}")
    print(f"{'num_stages':>11}{'smem/block':>12}{'fits?':>8}{'blocks/SM':>11}")
    for ns in sorted(BWD_SMEM[head_dim]):
        smem = BWD_SMEM[head_dim][ns]
        fits = smem <= SMEM_PER_BLOCK
        occ = str(blocks_per_sm(head_dim, ns, SMEM_PER_SM)) if fits else "-"
        print(f"{ns:>11}{smem:>12}{('yes' if fits else 'NO'):>8}{occ:>11}")
    print()


### Read the `blocks/SM` column

At `HEAD_DIM=64` it goes `2, 2, 2, 1, 1, 1`. The drop happens between `num_stages=3` (47,616 B) and
`num_stages=4` (51,968 B) — because two blocks fit on an SM only while each uses at most
`102,400 / 2 = 51,200` bytes.

Crossing that line **halves the SM's occupancy**: one resident block instead of two, so nothing is
left to run while a block stalls on a global load. Here is what it costs
(backward TFLOP/s, `BATCH=4, H=16, N_CTX=4096`):

| `num_stages` | smem | blocks/SM | causal=False | causal=True |
|---|---|---|---|---|
| 1 | 40,960 | 2 | 62.8 | 57.4 |
| 2 | 43,264 | 2 | 67.1 | 61.7 |
| **3** | **47,616** | **2** | **67.9** ← best | **61.6** |
| 4 | 51,968 | 1 | 55.2 | 51.3 |
| 5 | 56,320 | 1 | 56.1 ← *upstream pins this* | 51.2 |
| 6 | 60,672 | 1 | 56.0 | 51.1 |

![More pipelining is not more speed](../course/figures/fig_t06d_stages.svg)

**`num_stages=3` is 21% faster than upstream's `num_stages=5` on this card**, and the two extra
stages of prefetch buy nothing: they cannot compensate for losing half the occupancy.

At `HEAD_DIM=128` the column is `1, 1, 1, 1` — no depth ever wins a second resident block, because
even a single stage costs 80 KB. Extra stages there are pure loss: they add shared memory *and*
register spills (298 spilled registers at `ns=1`, 354 at `ns=4`) for no occupancy gain. So **the
fastest choice is the shallowest one**.

Which gives us a rule with a reason behind it.


In [ ]:
def occupancy_stages(head_dim, smem_per_sm, smem_per_block):
    """Pick num_stages for _attn_bwd: deepest pipeline that still keeps two blocks resident per SM.

    If two blocks can never be resident (HEAD_DIM=128 on a 100 KB card), extra pipeline stages buy
    no overlap and only cost shared memory and registers -> take the shallowest that fits.
    """
    table = BWD_SMEM[head_dim]
    fits = [ns for ns in sorted(table) if table[ns] <= smem_per_block]
    if not fits:
        raise ValueError(f"no num_stages fits HEAD_DIM={head_dim} in {smem_per_block} B")
    two_resident = [ns for ns in fits if smem_per_sm // table[ns] >= 2]
    return max(two_resident) if two_resident else min(fits)


CARDS = {                       # (smem per SM, smem per thread block), from the CUDA tuning guides
    "Ada / GA10x (cc 8.6, 8.9)": (102400, 101376),
    "A100 (cc 8.0)": (167936, 166912),
    "H100 (cc 9.0)": (233472, 232448),
}

print(f"{'card':<28}{'HEAD_DIM=64':>13}{'HEAD_DIM=128':>14}")
print("-" * 55)
for name, (per_sm, per_block) in CARDS.items():
    picks = [occupancy_stages(d, per_sm, per_block) for d in (64, 128)]
    print(f"{name:<28}{picks[0]:>13}{picks[1]:>14}")

ada = CARDS["Ada / GA10x (cc 8.6, 8.9)"]
assert occupancy_stages(64, *ada) == 3, "Ada keeps two blocks resident up to ns=3"
assert occupancy_stages(128, *ada) == 1, "Ada can never get two blocks at HEAD_DIM=128"
assert occupancy_stages(128, *CARDS["H100 (cc 9.0)"]) == 5, "on Hopper the rule reproduces ns=5"
print("\nPASS")
print("\nNote the bottom-right cell: on an H100 the rule picks 5 at HEAD_DIM=128 —")
print("exactly the constant upstream hardcodes. The tutorial is tuned for the GPU its authors had.")


### Now `HEAD_DIM=128` runs

Feed the rule our own card's numbers and re-run §8's check at the head dim that failed in §10.
Pipeline depth changes *when* tiles arrive, never *what* is computed — so the gradients should be
exactly what `num_stages=5` would have produced, had it been able to launch.


In [ ]:
if HAS_TRITON:
    NS = {d: occupancy_stages(d, SMEM_PER_SM, SMEM_PER_BLOCK) for d in (64, 128)}
    print(f"this card picks: HEAD_DIM=64 -> num_stages={NS[64]},  "
          f"HEAD_DIM=128 -> num_stages={NS[128]}\n")
    print(f"{'N_CTX':>6} {'D':>4} {'ns':>3} {'causal':>7} "
          f"{'max|dQ err|':>12} {'max|dK err|':>12} {'max|dV err|':>12}  status")
    worst = 0.0
    for D_ in (64, 128):
        for causal in (False, True):
            q, k, v = make_qkv(1, 2, 1024, D_)
            dout = torch.randn_like(q)
            _, dq, dk, dv = grads_of(attention, q, k, v, causal, 0.5, dout, True, True, True, NS[D_])
            _, dq_r, dk_r, dv_r = grads_of(reference_attention, q, k, v, causal, 0.5, dout)
            errs = [(a - b).abs().max().item() for a, b in [(dq, dq_r), (dk, dk_r), (dv, dv_r)]]
            worst = max(worst, max(errs))
            print(f"{1024:>6} {D_:>4} {NS[D_]:>3} {str(causal):>7} " +
                  " ".join(f"{e:>12.3e}" for e in errs) + f"  {'OK' if max(errs) < 1e-2 else 'FAIL'}")
            del q, k, v, dout
            torch.cuda.empty_cache()
    assert worst < 1e-2
    print(f"\nPASS — worst {worst:.3e}. The HEAD_DIM=128 backward now launches, and is correct.")
else:
    print("Recorded on RTX 4080 SUPER:")
    print("this card picks: HEAD_DIM=64 -> num_stages=3,  HEAD_DIM=128 -> num_stages=1\n")
    print(" N_CTX    D  ns  causal   max|dQ err|  max|dK err|  max|dV err|  status")
    print("  1024   64   3   False     4.883e-04    4.883e-04    7.324e-04  OK")
    print("  1024   64   3    True     1.953e-03    1.953e-03    1.953e-03  OK")
    print("  1024  128   1   False     1.465e-03    1.953e-03    1.465e-03  OK")
    print("  1024  128   1    True     3.906e-03    4.395e-03    3.906e-03  OK")
    print("\nPASS — worst 4.395e-03. The HEAD_DIM=128 backward now launches, and is correct.")


## 12. How fast is it, really?

Against `torch.nn.functional.scaled_dot_product_attention`, which dispatches to PyTorch's own
hand-tuned Flash kernels — forward *and* backward. Same shape as the tutorial's benchmark:
`BATCH=4, H=32, HEAD_DIM=64`, fp16.

The backward does roughly **2.5×** the forward's FLOPs (five matmuls against two), so we count
`2 · 2 · BATCH · H · N_CTX² · HEAD_DIM · 2.5`, halved when causal.


In [ ]:
if HAS_TRITON:
    import torch.nn.functional as F

    BATCH, NH, HD = 4, 32, 64
    NS64 = occupancy_stages(64, SMEM_PER_SM, SMEM_PER_BLOCK)

    def tflops(ms, N, causal, mult):
        f = 2.0 * 2.0 * BATCH * NH * N * N * HD * mult
        if causal:
            f *= 0.5
        return f * 1e-12 / (ms * 1e-3)

    for causal in (False, True):
        print(f"\ncausal={causal}   (TFLOP/s, higher is better)")
        print(f"{'N_CTX':>7} {'fwd':>8} {'bwd ns=5':>10} {'bwd ns=' + str(NS64):>10} {'gain':>7} | "
              f"{'sdpa fwd':>9} {'sdpa bwd':>9} | {'bwd vs sdpa':>12}")
        for N in [1024, 2048, 4096]:
            q, k, v = make_qkv(BATCH, NH, N, HD, seed=7)
            dout = torch.randn_like(q)
            ms_f = triton.testing.do_bench(lambda: attention(q, k, v, causal, 1.3))
            o5 = attention(q, k, v, causal, 1.3, True, True, True, 5)
            ms_b5 = triton.testing.do_bench(lambda: o5.backward(dout, retain_graph=True))
            o3 = attention(q, k, v, causal, 1.3, True, True, True, NS64)
            ms_b3 = triton.testing.do_bench(lambda: o3.backward(dout, retain_graph=True))
            ms_sf = triton.testing.do_bench(
                lambda: F.scaled_dot_product_attention(q, k, v, is_causal=causal, scale=1.3))
            os_ = F.scaled_dot_product_attention(q, k, v, is_causal=causal, scale=1.3)
            ms_sb = triton.testing.do_bench(lambda: os_.backward(dout, retain_graph=True))

            b5, b3 = tflops(ms_b5, N, causal, 2.5), tflops(ms_b3, N, causal, 2.5)
            sb = tflops(ms_sb, N, causal, 2.5)
            print(f"{N:>7} {tflops(ms_f, N, causal, 1.0):>8.1f} {b5:>10.1f} {b3:>10.1f} "
                  f"{(b3 / b5 - 1) * 100:>6.1f}% | {tflops(ms_sf, N, causal, 1.0):>9.1f} {sb:>9.1f} | "
                  f"{b3 / sb:>11.2f}x")
            del q, k, v, dout, o5, o3, os_
            torch.cuda.empty_cache()
else:
    print("Recorded on RTX 4080 SUPER (sm_89), fp16, BATCH=4 H=32 HEAD_DIM=64:\n")
    print("causal=False   (TFLOP/s)")
    print("  N_CTX      fwd   bwd ns=5   bwd ns=3    gain |  sdpa fwd  sdpa bwd | bwd vs sdpa")
    print("   1024    100.1       46.3       55.6   19.9% |      98.2      66.0 |       0.84x")
    print("   2048    103.1       51.3       63.2   23.3% |     102.1      77.8 |       0.81x")
    print("   4096    103.8       56.3       68.1   20.9% |     103.2      87.8 |       0.78x")
    print("\ncausal=True   (TFLOP/s)")
    print("  N_CTX      fwd   bwd ns=5   bwd ns=3    gain |  sdpa fwd  sdpa bwd | bwd vs sdpa")
    print("   1024     72.4       35.3       41.2   16.7% |      73.2      43.9 |       0.94x")
    print("   2048     84.4       44.4       52.9   19.2% |      89.0      58.6 |       0.90x")
    print("   4096     91.7       51.6       62.3   20.7% |      96.7      73.4 |       0.85x")


### Interpret: the forward keeps up, the backward does not

![The forward stays within a few percent of SDPA; the backward reaches 0.78-0.94x](../course/figures/fig_t06d_bench.svg)

Two honest readings.

**The `num_stages` fix is worth 17–23%**, at every length, in both modes, for free. It is one integer.

**Even fixed, the backward reaches only 0.78–0.94× of `torch`'s.** Put that next to the forward,
which is roughly at parity — it *edges* SDPA by 1–2% non-causal, and trails it by 3–5% causal (06c
measured the same picture). The backward is behind in both modes, and the gap *widens* with `N_CTX`.
Where does it go? Largely to the same shared-memory pressure §11 exposed: at `HEAD_DIM=64, ns=3` we
get two blocks per SM, while PyTorch's kernel — built on
[FlashAttention](https://github.com/Dao-AILab/flash-attention)'s hand-written CUDA — uses smaller
tiles, keeps more blocks resident, and spills far fewer registers (we spill 50–80 at `HEAD_DIM=64`).

Note we come *closest* when `causal=True` (0.94× at `N_CTX=1024`). The causal backward is exactly
where §6's load balancing pays: our programs are perfectly even, and the hand-written kernel has to
work for that.

The lesson is not "Triton is slow." The forward proves otherwise. It is that **the backward is a
harder kernel** — five matmuls, two reduction axes, and a shared-memory budget that decides your
occupancy before you write a line.

> Benchmarks on a warm GPU drift. Run the cell above twice: in isolation these numbers are stable to
> under 1%, but the first `N_CTX` you touch after a long notebook session can read several percent
> low. The figure's values were measured in a fresh process, one shape at a time.


## 13. What you actually learned about your GPU

06c ended with a ledger of deletions. This chapter has nothing to delete, so its ledger is of
*assumptions* — the ones the backward makes about your hardware without ever mentioning it.

| Assumption in the tutorial's backward | Where it lives | On Ampere/Ada |
|---|---|---|
| `num_stages=5` will fit in shared memory | `_attention.backward`, a bare constant | **false at HEAD_DIM=128** (§10) |
| deeper pipelining is faster | implicit | **false** — costs 21% at HEAD_DIM=64 (§11) |
| `N_CTX % 128 == 0` | no boundary masking anywhere | true only if you pad |
| `q, k, v, o, do` are contiguous and identically strided | `_attn_bwd_preprocess`'s index math | true for the tutorial's tensors |
| `BLOCK_N1 % BLOCK_M1 == 0`, `BLOCK_M2 % BLOCK_N2 == 0` | two `tl.static_assert`s | enforced at compile time |
| fp16 in, fp32 accumulate | `.to(tl.float32)` on `do`, `dp` | correct, and load-bearing (§4) |

And the two facts worth carrying out of this notebook:

1. **A kernel with no architecture branches can still be architecture-specific.** The backward names
   no GPU. It cannot run on half of them. Resources are an interface too.
2. **`num_stages` is an occupancy knob, not a speed knob.** Compute `smem_per_sm // smem_per_block`
   before you tune anything else.


> **Going to production.** What you have now is a correct, autograd-ready fused attention — forward
> from 06c, backward from here — at roughly 0.8× of `torch`'s on the backward. To go further:
>
> - **Ragged sequences.** Every `tl.load` here is unmasked; `N_CTX` must be a multiple of 128. Add
>   `mask=`/`other=0.0` and mask the `qk` tail, or pad. The backward needs it in three places
>   (`preprocess`, `dkdv`, `dq`), not one.
> - **Autotune the backward.** Upstream never does — `num_stages`, `num_warps` and all four block
>   sizes are bare constants. §11's rule is a *derivation*, not a search; a real `@triton.autotune`
>   over `(BLOCK_M1, BLOCK_N1, num_warps, num_stages)` keyed on `(N_CTX, HEAD_DIM)` would beat it,
>   and would have caught the `HEAD_DIM=128` failure at import instead of at launch.
> - **Register spills.** We spill 50–80 registers at `HEAD_DIM=64` and ~300 at `128`. Check
>   `kernel.n_spills`; shrinking `BLOCK_N1` to 64 trades tile efficiency for register room.
> - **Bigger head dims.** At `HEAD_DIM=256` even `num_stages=1` will not fit 99 KB at these block
>   sizes. You must shrink `BLOCK_N1`. Exercise B has you predict exactly where that wall is.
> - **Real deployment.** Use [FlashAttention](https://github.com/Dao-AILab/flash-attention) or
>   `F.scaled_dot_product_attention`. They carry the Hopper/Blackwell paths, dropout, ALiBi, GQA, and
>   variable-length batching. Build this to *understand* them; ship theirs.


## Exercises

Five, in rising order of difficulty. All are **pure NumPy/Python** — they run whether or not you have
a GPU. Each has a collapsible solution; try it before you peek.


### Exercise A — audit a kernel for architecture gates

Write `count_gates(source, tokens)` returning a `dict[str, int]` of how many times each token appears
in `source`, **omitting tokens that never appear**. This is §1's tool, in three lines.

Note that we count *occurrences*, not lines: one `if is_blackwell()` can gate many call sites, and
`warp_specialize` shows up three times in four lines of the forward.

**Predict first:** applied to the backward's source, how many keys will the dict have?


In [ ]:
GATES = ["is_hip", "is_cuda", "is_hopper", "is_blackwell", "warp_specialize", "float8e5"]

FWD_SNIPPET = """
    if is_hip():
        extra_kern_args = {"waves_per_eu": waves_per_eu}
    dtype = tl.float8e5 if FP8_OUTPUT else tl.float16
    for start_n in tl.range(lo, hi, BLOCK_N, warp_specialize=warp_specialize):
        if is_blackwell() and warp_specialize:
            pass
"""

BWD_SNIPPET = """
    qT_ptrs = Q + offs_m[None, :] * stride_tok + offs_k[:, None] * stride_d
    qkT = tl.dot(k, qT)
    pT = tl.math.exp2(qkT - m[None, :])
    dk += tl.dot(dsT, tl.trans(qT))
"""


def count_gates(source, tokens):
    raise NotImplementedError("your turn")


fwd = count_gates(FWD_SNIPPET, GATES)
bwd = count_gates(BWD_SNIPPET, GATES)
print("forward snippet :", fwd)
print("backward snippet:", bwd)

assert fwd == {"is_hip": 1, "is_blackwell": 1, "warp_specialize": 3, "float8e5": 1}, fwd
assert bwd == {}, "the backward names no architecture"
assert all(v > 0 for v in fwd.values()), "absent tokens must be omitted, not stored as 0"
print("\nPASS — the backward's gate dict is empty. There is nothing to strip.")


<details>
<summary>▶ Show solution</summary>

```python
def count_gates(source, tokens):
    counts = {t: source.count(t) for t in tokens}
    return {t: n for t, n in counts.items() if n}

# `warp_specialize` scores 3: the tl.range kwarg name, its value, and the
# `is_blackwell() and warp_specialize` condition. Counting occurrences rather than
# lines is the point -- a single gate predicate can steer many call sites.
#
# §1 used re.findall(re.escape(t), region), which equals str.count for plain tokens
# but also survives tokens containing regex metacharacters.
```

</details>

### Exercise B — where is the shared-memory wall?

`_attn_bwd`'s footprint grows with `HEAD_DIM` and `num_stages`. §11 measured two head dims; the
pattern is exactly linear in both. Fit it.

Write `smem_model(head_dim, num_stages)` reproducing `BWD_SMEM` **exactly**, from this observed
structure:

- at `num_stages=1` the cost is `640 · head_dim` bytes;
- going `1 → 2` adds a flat `2304` bytes, independent of `head_dim`;
- **each** stage beyond the second adds `64 · head_dim + 256` bytes.

Then answer the real question with `max_head_dim(smem_per_block)`: the largest `head_dim` in
`{64, 128, 256}` that fits a given per-block budget at `num_stages=1`.

**Predict first:** does `HEAD_DIM=256` fit in Ada's 99 KB? In Hopper's 227 KB?


In [ ]:
BWD_SMEM = {
    64:  {1: 40960, 2: 43264, 3: 47616, 4: 51968, 5: 56320, 6: 60672},
    128: {1: 81920, 2: 84224, 3: 92672, 4: 101120, 5: 109568, 6: 118016},
}
ADA_PER_BLOCK, HOPPER_PER_BLOCK = 101376, 232448


def smem_model(head_dim, num_stages):
    raise NotImplementedError("your turn")


def max_head_dim(smem_per_block):
    raise NotImplementedError("your turn")


for hd, table in BWD_SMEM.items():
    for ns, measured in table.items():
        got = smem_model(hd, ns)
        assert got == measured, f"HEAD_DIM={hd} ns={ns}: model {got} != measured {measured}"
print("model reproduces all 12 measured values.\n")

need = smem_model(256, 1)
print(f"HEAD_DIM=256, num_stages=1 needs {need:,} B ({need / 1024:.0f} KB)")
print(f"  Ada    ( 99 KB = {ADA_PER_BLOCK:,} B): {'fits' if need <= ADA_PER_BLOCK else 'NO'}")
print(f"  Hopper (227 KB = {HOPPER_PER_BLOCK:,} B): {'fits' if need <= HOPPER_PER_BLOCK else 'NO'}")

assert max_head_dim(ADA_PER_BLOCK) == 128
assert max_head_dim(HOPPER_PER_BLOCK) == 256
print("\nPASS — at these block sizes an Ada card cannot run HEAD_DIM=256 at any pipeline depth,")
print("because the footprint only grows with num_stages.")


<details>
<summary>▶ Show solution</summary>

```python
def smem_model(head_dim, num_stages):
    smem = 640 * head_dim                              # num_stages = 1
    if num_stages >= 2:
        smem += 2304                                   # flat, independent of head_dim
    if num_stages >= 3:
        smem += (64 * head_dim + 256) * (num_stages - 2)
    return smem


def max_head_dim(smem_per_block):
    fits = [hd for hd in (64, 128, 256) if smem_model(hd, 1) <= smem_per_block]
    return max(fits) if fits else 0

# HEAD_DIM=256 at num_stages=1 needs 640*256 = 163_840 B = 160 KB. Ada gives you 99 KB;
# Hopper gives 227 KB. And since the model is monotonically increasing in num_stages,
# "does not fit at ns=1" means "does not fit at any depth" -- so on Ada, HEAD_DIM=256 is
# unreachable at these block sizes, no matter how you tune the pipeline.
#
# That is why real kernels shrink BLOCK_N1 as HEAD_DIM grows, rather than shrinking
# num_stages forever: the 640*head_dim base scales with the TILE, not with the pipeline.
```

</details>

### Exercise C — the occupancy rule, from scratch

Reimplement §11's rule. Write `occupancy_stages(head_dim, smem_per_sm, smem_per_block)`:

1. keep only pipeline depths whose footprint fits `smem_per_block`; raise `ValueError` if none do;
2. among those, keep the ones where **at least two blocks** stay resident per SM
   (`smem_per_sm // footprint >= 2`);
3. if any qualify, return the **deepest**; otherwise return the **shallowest** that fits.

**Predict first:** the B200 has the same 227 KB / 228 KB budget as an H100. Does it pick the same
`num_stages` at `HEAD_DIM=128`? What does that tell you about where upstream's constant came from?


In [ ]:
CARDS = {                            # (smem per SM, smem per thread block)
    "Ada / GA10x": (102400, 101376),
    "A100": (167936, 166912),
    "H100": (233472, 232448),
    "B200": (233472, 232448),
}


def occupancy_stages(head_dim, smem_per_sm, smem_per_block):
    raise NotImplementedError("your turn")


print(f"{'card':<14}{'HEAD_DIM=64':>13}{'HEAD_DIM=128':>14}")
for name, (per_sm, per_block) in CARDS.items():
    print(f"{name:<14}{occupancy_stages(64, per_sm, per_block):>13}"
          f"{occupancy_stages(128, per_sm, per_block):>14}")

assert occupancy_stages(64, *CARDS["Ada / GA10x"]) == 3
assert occupancy_stages(128, *CARDS["Ada / GA10x"]) == 1
assert occupancy_stages(128, *CARDS["H100"]) == 5      # == upstream's hardcoded constant
assert occupancy_stages(128, *CARDS["B200"]) == 5

try:                                                   # an absurdly small budget fits nothing
    occupancy_stages(64, 1024, 1024)
    raise AssertionError("should have raised ValueError")
except ValueError:
    print("\nraises ValueError when nothing fits.")
print("PASS")


<details>
<summary>▶ Show solution</summary>

```python
def occupancy_stages(head_dim, smem_per_sm, smem_per_block):
    table = BWD_SMEM[head_dim]
    fits = [ns for ns in sorted(table) if table[ns] <= smem_per_block]
    if not fits:
        raise ValueError(f"no num_stages fits HEAD_DIM={head_dim} in {smem_per_block} B")
    two_resident = [ns for ns in fits if smem_per_sm // table[ns] >= 2]
    return max(two_resident) if two_resident else min(fits)

# On H100 and B200 the rule returns 5 at HEAD_DIM=128 -- upstream's hardcoded NUM_STAGES.
# That is not a coincidence. `num_stages=5` is not a universal constant; it is the correct
# answer for a 228 KB SM, baked into a tutorial that also runs on 100 KB cards.
#
# Step 3's fallback carries the weight: at HEAD_DIM=128 on Ada no depth reaches two blocks,
# so deeper pipelining cannot buy overlap -- it only spends shared memory and registers.
# Take the shallowest. That is why the rule returns 1 there, not 4 (the largest that fits).
```

</details>

### Exercise D — diagnose the deleted line from the ratio

You are reviewing a colleague's fused-attention backward. Their gradients are wrong. All they give
you is the **median ratio** of each broken gradient to the correct one.

Write `diagnose(dq_ratio, dk_ratio, dv_ratio, sm_scale)` returning one of:

- `"nothing wrong"` — all three ratios are 1;
- `"dq *= LN2"` — dQ is off by `1/ln 2`, dK and dV clean;
- `"dk *= sm_scale"` — dK is off by `1/sm_scale`, dQ and dV clean;
- `"arg_k prescale"` — anything else (a semantic bug, §9).

Use a tolerance of `1e-3` on each ratio. Think about the order of your checks.

**Predict first:** `dV` is touched by neither epilogue scale. What does a wrong `dV` prove?


In [ ]:
import math

LN2 = 0.6931471824645996


def diagnose(dq_ratio, dk_ratio, dv_ratio, sm_scale):
    raise NotImplementedError("your turn")


# the four rows measured in §9, sm_scale = 0.5
assert diagnose(1.0000, 1.0000, 1.0000, 0.5) == "nothing wrong"
assert diagnose(1.4427, 1.0000, 1.0000, 0.5) == "dq *= LN2"
assert diagnose(1.0000, 2.0000, 1.0000, 0.5) == "dk *= sm_scale"
assert diagnose(3.7585, 3.4710, 1.6180, 0.5) == "arg_k prescale"

# ... and it must generalise to another sm_scale
assert diagnose(1.0, 1.0 / 0.125, 1.0, 0.125) == "dk *= sm_scale"
assert diagnose(1.0 / LN2, 1.0, 1.0, 0.125) == "dq *= LN2"

# a broken dV alone is already conclusive
assert diagnose(1.0, 1.0, 1.6180, 0.5) == "arg_k prescale"
print("PASS — you can name the deleted line from three numbers.")
print(f"\n1/ln2 = {1 / LN2:.4f} is the same for every sm_scale;")
print("1/sm_scale is not — which is how you tell the two epilogue bugs apart.")


<details>
<summary>▶ Show solution</summary>

```python
import math

LN2 = 0.6931471824645996


def diagnose(dq_ratio, dk_ratio, dv_ratio, sm_scale):
    tol = 1e-3
    clean = lambda r: math.isclose(r, 1.0, abs_tol=tol)

    if clean(dq_ratio) and clean(dk_ratio) and clean(dv_ratio):
        return "nothing wrong"

    # dV is touched by neither epilogue scale, so a wrong dV means the softmax itself is
    # wrong. Likewise, the epilogues are independent -- each can break only ONE gradient,
    # so two broken gradients cannot be an epilogue bug either.
    n_broken = sum(not clean(r) for r in (dq_ratio, dk_ratio, dv_ratio))
    if not clean(dv_ratio) or n_broken > 1:
        return "arg_k prescale"

    if math.isclose(dq_ratio, 1.0 / LN2, abs_tol=tol):
        return "dq *= LN2"
    if math.isclose(dk_ratio, 1.0 / sm_scale, abs_tol=tol):
        return "dk *= sm_scale"
    return "arg_k prescale"        # a lone broken gradient, but by the wrong factor

# What a wrong dV proves: dV = P^T dO, and neither `dq *= LN2` nor `dk *= sm_scale` appears
# anywhere in that expression. So dV can only move if P moved -- i.e. if the argument of
# exp2 changed, which is precisely what dropping the arg_k prescale does. `dV` is the
# cheapest single check for "is my softmax right?" in the whole backward.
```

</details>

### Exercise E — prove the causal backward is load-balanced

§6 showed every program runs the same number of inner-loop steps at `N_CTX=1024`. Prove it holds for
**any** `N_CTX` that is a multiple of `BLOCK_N1`.

Write `bwd_steps(pid, N_CTX, causal)` returning the four counts (`dkdv_masked`, `dkdv_bulk`,
`dq_masked`, `dq_bulk`) exactly as `_attn_bwd` computes them, with upstream's constants
`BLOCK_M1, BLOCK_N1, BLOCK_M2, BLOCK_N2 = 32, 128, 128, 32` and `BLK_SLICE_FACTOR = 2`.

The asserts then check the total is independent of `pid`, and matches the closed form
`16 + (N_CTX − 128)/32`.

**Predict first:** the forward's causal work grows with `start_m` (06c, Exercise C). What cancels it
here?


In [ ]:
BLOCK_M1, BLOCK_N1, BLOCK_M2, BLOCK_N2 = 32, 128, 128, 32
BLK_SLICE_FACTOR = 2
MASK_BLOCK_M1 = BLOCK_M1 // BLK_SLICE_FACTOR
MASK_BLOCK_N2 = BLOCK_N2 // BLK_SLICE_FACTOR


def bwd_steps(pid, N_CTX, causal):
    raise NotImplementedError("your turn")


for N in (256, 1024, 4096):
    totals = {sum(bwd_steps(pid, N, True).values()) for pid in range(N // BLOCK_N1)}
    assert len(totals) == 1, f"N_CTX={N}: programs are NOT balanced, totals={sorted(totals)}"
    closed_form = 16 + (N - BLOCK_N1) // BLOCK_M1
    assert totals == {closed_form}, f"N_CTX={N}: got {totals}, closed form says {closed_form}"
    print(f"N_CTX={N:>5}: every one of {N // BLOCK_N1:>2} programs runs "
          f"{closed_form:>3} steps  (= 16 + (N_CTX-128)/32)")

# non-causal: every program sweeps the full axis on both halves
assert bwd_steps(0, 1024, False) == {"dkdv_masked": 0, "dkdv_bulk": 32,
                                     "dq_masked": 0, "dq_bulk": 32}
print("\nPASS — causal work is constant across programs; non-causal is trivially uniform.")


<details>
<summary>▶ Show solution</summary>

```python
def bwd_steps(pid, N_CTX, causal):
    if not causal:
        return {"dkdv_masked": 0, "dkdv_bulk": N_CTX // BLOCK_M1,
                "dq_masked": 0, "dq_bulk": N_CTX // BLOCK_N2}

    # this program owns KEY block `pid`: the diagonal, then every query block after it
    start_n = pid * BLOCK_N1
    dkdv_masked = BLOCK_N1 // MASK_BLOCK_M1                  # 128 // 16 = 8
    start_m = start_n + dkdv_masked * MASK_BLOCK_M1          # = start_n + BLOCK_N1
    dkdv_bulk = (N_CTX - start_m) // BLOCK_M1

    # the SAME program owns QUERY block `pid`: the diagonal, then every key block before it
    start_m2 = pid * BLOCK_M2
    dq_masked = BLOCK_M2 // MASK_BLOCK_N2                    # 128 // 16 = 8
    end_n = start_m2 + BLOCK_M2 - dq_masked * MASK_BLOCK_N2  # = start_m2
    dq_bulk = end_n // BLOCK_N2

    return {"dkdv_masked": dkdv_masked, "dkdv_bulk": dkdv_bulk,
            "dq_masked": dq_masked, "dq_bulk": dq_bulk}

# The closed form. With BLOCK_M1 == BLOCK_N2 == 32 and BLOCK_N1 == BLOCK_M2 == 128:
#
#   dkdv_bulk = (N_CTX - 128*pid - 128) / 32   -> falls by 4 per pid
#   dq_bulk   = (128*pid) / 32                 -> rises by 4 per pid
#   sum       = (N_CTX - 128) / 32             -> independent of pid
#
# plus 8 + 8 masked steps, giving 16 + (N_CTX - 128)/32.
#
# What cancels the forward's imbalance: key block `pid` is read by every LATER query, and
# query block `pid` reads every EARLIER key. The two triangles are reflections of one
# another, so handing both jobs to one program fills the gap exactly. The forward has only
# one triangle and no partner to pair with -- hence its 8x spread across programs.
```

</details>

## Further Reading

**Source of truth**

- The tutorial this notebook runs: [`06-fused-attention.py`](https://triton-lang.org/main/getting-started/tutorials/06-fused-attention.html)
  (OpenAI Triton kernel team) — and its local copy, `tutorials_jupyter/06-fused-attention.ipynb`.
- [CUDA Programming Guide — Technical Specifications per Compute Capability](https://docs.nvidia.com/cuda/cuda-programming-guide/05-appendices/compute-capabilities.html)
  — the "maximum amount of shared memory per thread block" row is the whole of §10.
- [Ampere Tuning Guide](https://docs.nvidia.com/cuda/ampere-tuning-guide/index.html) — cc 8.0 (A100):
  *"shared memory capacity per SM is 164 KB"*, 163 KB per block; cc 8.6 (GA10x): 100 KB / **99 KB**.
- [Ada Tuning Guide](https://docs.nvidia.com/cuda/ada-tuning-guide/index.html) — *"The shared memory
  capacity per SM is 100 KB"*, *"the maximum shared memory per thread block is 99 KB."*
- [Blackwell Tuning Guide](https://docs.nvidia.com/cuda/blackwell-tuning-guide/index.html) — cc 10.0:
  228 KB / 227 KB; cc 12.0 (GeForce): 128 KB / **99 KB**.
- [`triton.Config`](https://triton-lang.org/main/python-api/generated/triton.Config.html) — what
  `num_stages` and `num_warps` actually control.


**Going deeper**

- FlashAttention-2: [arXiv:2307.08691](https://arxiv.org/abs/2307.08691) — Algorithm 2 is the backward
  this notebook runs; §3.1 explains the two-axis split that §6 measures.
- The original FlashAttention: [arXiv:2205.14135](https://arxiv.org/abs/2205.14135).
- [FlashAttention (Dao-AILab)](https://github.com/Dao-AILab/flash-attention) — the hand-written CUDA
  backward we lose to in §12, by 6–22%.
- [CUDA Occupancy Calculator](https://docs.nvidia.com/cuda/cuda-occupancy-calculator/index.html) —
  the tool that automates §11's `smem_per_sm // smem_per_block`.
- [Using Shared Memory in CUDA](https://developer.nvidia.com/blog/using-shared-memory-cuda-cc/)
  (NVIDIA blog) — why occupancy and shared memory trade off.

**In this course**

- **06a** — the forward pass, unpacked (online softmax, causal `STAGE`).
- **06b** — the backward pass, unpacked on paper. Everything this notebook *runs*, 06b *derives*.
- **06c** — stripping the forward to CUDA + Ampere/Ada, and proving TMA is free there.
- **Chapter 16b** — the same fused attention in raw CUDA C++.
- **Chapters 16c / 16d** — FlashAttention-3 and -4, built on the Hopper/Blackwell machinery whose
  shared-memory budget §10 shows the tutorial silently assumed.


## Recap — the architecture you can't see

You took 06b's paper backward, made it run, and then found the thing neither 06b nor 06c could have
told you: **the tutorial's backward does not work on the GPU most people own.**

| What you did | Where | The evidence |
|---|---|---|
| Audited the backward for arch gates | §1 | 297 lines, **0** references — the strip is empty |
| Launched `_attn_bwd_preprocess` | §4 | `D` matches torch exactly; the fp32 upcast is load-bearing |
| Measured the causal load balance | §6 | every program runs `16 + (N_CTX−128)/32` steps |
| Checked all three gradients | §8, §11 | worst error **4.4e-3** vs torch autograd (tol 1e-2) |
| Broke each scaling on purpose | §9 | missing `dq *= LN2` ⇒ dQ **×1.4427**; missing `dk *= sm_scale` ⇒ dK **×2.0** |
| Hit the shared-memory wall | §10 | `HEAD_DIM=128` needs **107 KB**; Ampere/Ada give **99 KB** |
| Found the occupancy cliff | §11 | `num_stages=3` keeps 2 blocks/SM and is **21% faster** than upstream's 5 |
| Benchmarked honestly | §12 | forward within ±5% of SDPA; backward reaches only **0.78–0.94×** |


**The transferable lesson.** 06c taught you to read a kernel's `if`s as compile-time switches and
delete the ones your card cannot take. This chapter is the other half of that skill, and the harder
one: **a kernel's most binding hardware assumptions are usually the ones it never writes down.**

`num_stages=5` is not a tuning preference. It is a claim — that your SM has Hopper's shared memory —
made by a bare integer, with no `if` around it, in a file whose backward mentions no architecture at
all. The compiler will not warn you. The type checker will not warn you. You find it by launching the
kernel, reading `Required: 109568, Hardware limit: 101376`, and doing the division.

So before you tune a kernel you did not write: **compute its shared memory per block, divide it into
your SM's, and look at the integer you get.** If it is 1 where it could be 2, no amount of pipelining
will save you. That one division was worth 21% here — and it is the same division whether the kernel
is Triton, CUTLASS, or hand-written PTX.

**Where to go next.** **Chapters 16c/16d** take up FlashAttention-3 and -4, which are largely *about*
spending Hopper's and Blackwell's much larger shared-memory budget well — 227 KB per block, and a
`tcgen05` unit to feed. Having watched 99 KB decide this chapter, you will read those designs
differently.
